# 3.9 神经符号 AI (Neurosymbolic AI)

> 🕐 预估学习时间：30分钟

神经符号 AI（Neurosymbolic AI）将神经网络的感知能力与符号推理的逻辑能力相结合，被认为是迈向通用人工智能的重要方向之一。

本节涵盖：
- 神经符号 AI 的基本概念与动机
- 神经-符号的多种集成方式
- 可微逻辑编程与模糊逻辑
- LLM 与符号推理的结合
- 知识图谱与神经推理
- 实践中的神经符号系统设计

## 1. 神经符号 AI 概述

**为什么需要神经符号 AI？**
- **神经网络的优势**：感知能力强、可从数据学习、对噪声鲁棒
- **神经网络的弱点**：缺乏可解释性、难以保证逻辑一致性、数据饥渴
- **符号推理的优势**：逻辑严谨、可解释、可组合、无需大量数据
- **符号推理的弱点**：难以处理感知输入、对噪声敏感、知识获取困难

**神经符号 AI 的核心思想**：用神经网络处理感知与模式识别，用符号系统执行逻辑推理，二者互补。

**历史与复兴**：
- 1950s-1980s：符号 AI 主导（专家系统、GOFAI）
- 1990s-2010s：连接主义复兴，深度学习崛起
- 2020s：神经符号 AI 复兴，LLM + 工具调用、Program-Aided Models 等兴起

**代表方向**：DeepProblog、Neural Theorem Provers、LLM+Tool、Program-Aided Language Models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

print('=== Neurosymbolic Framework ===')


class NeuralPerception(nn.Module):
    def __init__(self, input_dim=64, hidden=128, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        return self.net(x)


class SymbolicReasoner:
    def __init__(self, rules):
        # rules: list of (premise_set, conclusion)
        self.rules = rules

    def infer(self, facts):
        # facts: set of symbol ids
        derived = set(facts)
        changed = True
        while changed:
            changed = False
            for premises, conclusion in self.rules:
                if premises.issubset(derived) and conclusion not in derived:
                    derived.add(conclusion)
                    changed = True
        return derived


class NeuroSymbolicModel(nn.Module):
    def __init__(self, input_dim=64, n_classes=10, rules=None):
        super().__init__()
        self.perception = NeuralPerception(input_dim=input_dim, n_classes=n_classes)
        self.reasoner = SymbolicReasoner(rules or [])
        self.class_to_symbol = {i: f'cls_{i}' for i in range(n_classes)}

    def forward(self, x):
        logits = self.perception(x)
        preds = logits.argmax(dim=-1)
        return logits, preds

    def reason(self, x):
        logits, preds = self.forward(x)
        results = []
        for p in preds.tolist():
            facts = {self.class_to_symbol[p]}
            derived = self.reasoner.infer(facts)
            results.append(derived)
        return logits, results


# Demo: image-like classification + symbolic reasoning
# Suppose classes 0..4 are animals, rules derive properties
rules = [
    ({'cls_0'}, 'is_mammal'),
    ({'cls_1'}, 'is_bird'),
    ({'cls_2'}, 'is_fish'),
    ({'is_mammal'}, 'has_fur'),
    ({'is_bird'}, 'has_feathers'),
    ({'is_fish'}, 'has_scales'),
    ({'is_mammal', 'has_fur'}, 'is_warm_blooded'),
    ({'is_bird', 'has_feathers'}, 'is_warm_blooded'),
]

model = NeuroSymbolicModel(input_dim=64, n_classes=5, rules=rules)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Synthetic training data
X = torch.randn(128, 64)
y = torch.randint(0, 5, (128,))

for epoch in range(20):
    logits, _ = model(X)
    loss = F.cross_entropy(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}')

# Inference with reasoning
X_test = torch.randn(3, 64)
logits, derived_facts = model.reason(X_test)
preds = logits.argmax(dim=-1).tolist()
print(f'\nPredicted classes: {preds}')
for i, facts in enumerate(derived_facts):
    print(f'Sample {i}: derived facts = {sorted(facts)}')

print(f'\nKey: Neurosymbolic models combine neural perception with interpretable symbolic rules.')
print(f'The neural part handles noisy input; the symbolic part enforces logical consistency.')

## 2. 神经-符号集成方式

**类型 1：神经网络作为特征提取器**
- 神经网络提取特征 → 符号系统基于特征做推理
- 优点：保留符号推理的可解释性
- 缺点：特征到符号的映射可能丢失信息

**类型 2：符号作为约束**
- 符号规则作为硬/软约束加在神经网络上
- 例如：逻辑约束层、约束损失项
- 优点：让神经网络满足领域知识

**类型 3：概率逻辑**
- 将符号规则概率化，与神经网络联合训练
- 例如：Markov Logic Networks、ProbLog
- 优点：可处理不确定性

**类型 4：端到端可微**
- 将符号操作改写为可微操作（模糊逻辑）
- 整个系统可反向传播训练
- 优点：联合优化、无需分阶段训练

In [ ]:
torch.manual_seed(42)

print('=== Neural-Symbolic Integration Approaches ===')


class NeuralFeatureExtractor(nn.Module):
    def __init__(self, input_dim=32, feat_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, feat_dim),
        )

    def forward(self, x):
        return self.encoder(x)


class LogicConstraintLayer(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.n_classes = n_classes
        # Constraint matrix: (i, j) means class i and class j are mutually exclusive
        self.register_buffer('mutex', torch.eye(n_classes))

    def forward(self, logits):
        probs = F.softmax(logits, dim=-1)
        # Soft mutual exclusion: penalize when multiple classes have high prob
        # entropy of distribution should be low (peaked)
        entropy = -(probs * (probs + 1e-8).log()).sum(dim=-1).mean()
        max_entropy = math.log(self.n_classes)
        # normalized entropy penalty (0 = peaked, 1 = uniform)
        penalty = entropy / max_entropy
        return probs, penalty

    def constraint_loss(self, logits):
        _, penalty = self.forward(logits)
        return penalty


class ConstrainedClassifier(nn.Module):
    def __init__(self, input_dim=32, n_classes=5, constraint_weight=0.1):
        super().__init__()
        self.extractor = NeuralFeatureExtractor(input_dim=input_dim, feat_dim=16)
        self.classifier = nn.Linear(16, n_classes)
        self.constraint = LogicConstraintLayer(n_classes=n_classes)
        self.constraint_weight = constraint_weight

    def forward(self, x):
        feats = self.extractor(x)
        logits = self.classifier(feats)
        probs, penalty = self.constraint(logits)
        return logits, probs, penalty

    def loss(self, x, y):
        logits, _, penalty = self.forward(x)
        ce_loss = F.cross_entropy(logits, y)
        return ce_loss + self.constraint_weight * penalty


model = ConstrainedClassifier(input_dim=32, n_classes=5, constraint_weight=0.2)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

X = torch.randn(200, 32)
y = torch.randint(0, 5, (200,))

print('Training constrained classifier...')
for epoch in range(30):
    loss = model.loss(X, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            logits, probs, penalty = model(X)
            acc = (logits.argmax(dim=-1) == y).float().mean().item()
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, acc={acc:.3f}, constraint_penalty={penalty.item():.4f}')

# Compare: with vs without constraint
model_unconstrained = ConstrainedClassifier(input_dim=32, n_classes=5, constraint_weight=0.0)
opt2 = torch.optim.AdamW(model_unconstrained.parameters(), lr=1e-3)
for epoch in range(30):
    loss = model_unconstrained.loss(X, y)
    opt2.zero_grad()
    loss.backward()
    opt2.step()

with torch.no_grad():
    _, probs_c, _ = model(X)
    _, probs_u, _ = model_unconstrained(X)
    ent_c = -(probs_c * (probs_c + 1e-8).log()).sum(dim=-1).mean().item()
    ent_u = -(probs_u * (probs_u + 1e-8).log()).sum(dim=-1).mean().item()
print(f'\nMean entropy - constrained: {ent_c:.4f}, unconstrained: {ent_u:.4f}')
print('Constrained model produces more peaked (confident) distributions.')

print(f'\nKey: Symbolic constraints regularize neural outputs toward logically consistent predictions.')
print(f'The constraint weight trades task accuracy for logical consistency.')

## 3. 可微逻辑编程

**核心思想**：将传统逻辑操作（AND、OR、NOT）替换为可微的模糊逻辑版本，使整个推理过程可反向传播。

**模糊逻辑操作**：
- 模糊 AND：$a \land b = \min(a, b)$ 或 $a \cdot b$（乘积 t-norm）
- 模糊 OR：$a \lor b = \max(a, b)$ 或 $a + b - a \cdot b$
- 模糊 NOT：$\lnot a = 1 - a$

**可微逻辑的优势**：
- 规则可以与神经网络联合训练
- 可从数据中学习规则权重
- 保留逻辑结构的同时处理不确定性

**代表工作**：
- DeepProblog：将 Prolog 与神经网络结合
- Logic Tensor Networks
- t-norm 模糊逻辑层

In [ ]:
torch.manual_seed(42)

print('=== Differentiable Logic Programming ===')


class DifferentiableLogic(nn.Module):
    def __init__(self):
        super().__init__()
        # Learnable sharpness for smooth min/max approximations
        self.sharpness = nn.Parameter(torch.tensor(2.0))

    def fuzzy_and(self, a, b):
        # Product t-norm (differentiable everywhere)
        return a * b

    def fuzzy_or(self, a, b):
        # Probabilistic sum
        return a + b - a * b

    def fuzzy_not(self, a):
        return 1.0 - a

    def smooth_min(self, a, b):
        # Soft-min approximation using negative softmax
        s = self.sharpness.abs() + 0.1
        stacked = torch.stack([a, b], dim=-1) * (-s)
        return torch.logsumexp(stacked, dim=-1) / (-s)

    def smooth_max(self, a, b):
        s = self.sharpness.abs() + 0.1
        stacked = torch.stack([a, b], dim=-1) * s
        return torch.logsumexp(stacked, dim=-1) / s


class LogicRule(nn.Module):
    def __init__(self, n_predicates, hidden=32):
        super().__init__()
        self.n_predicates = n_predicates
        # Learnable attention over predicates for premise
        self.premise_net = nn.Sequential(
            nn.Linear(n_predicates, hidden), nn.ReLU(),
            nn.Linear(hidden, n_predicates),
        )
        # Learnable head: which conclusion predicate to activate
        self.conclusion_net = nn.Sequential(
            nn.Linear(n_predicates, hidden), nn.ReLU(),
            nn.Linear(hidden, n_predicates),
        )
        self.logic = DifferentiableLogic()

    def forward(self, predicates):
        # predicates: (batch, n_predicates) in [0, 1]
        premise_weights = torch.sigmoid(self.premise_net(predicates))
        # Fuzzy AND over weighted premises (product t-norm reduction)
        premise_truth = (predicates * premise_weights).prod(dim=-1, keepdim=True)
        conclusion_logits = self.conclusion_net(predicates)
        conclusion = torch.sigmoid(conclusion_logits)
        # If premise is true, activate conclusion
        activated = self.logic.fuzzy_or(
            predicates,
            premise_truth * conclusion,
        )
        return activated


class DifferentiableLogicProgram(nn.Module):
    def __init__(self, n_predicates=6, n_rules=3):
        super().__init__()
        self.rules = nn.ModuleList([
            LogicRule(n_predicates) for _ in range(n_rules)
        ])
        self.n_predicates = n_predicates

    def forward(self, predicates, steps=2):
        current = predicates
        for _ in range(steps):
            for rule in self.rules:
                current = rule(current)
            current = torch.clamp(current, 0.0, 1.0)
        return current


# Demo: learn logic rules from data
# Suppose we want the program to learn: if p0 and p1 then p4; if p2 then p5
program = DifferentiableLogicProgram(n_predicates=6, n_rules=3)
optimizer = torch.optim.AdamW(program.parameters(), lr=5e-3)

# Training data: input predicates -> expected output predicates
n_samples = 256
X_train = torch.rand(n_samples, 6)
# Target: p4 = X[:,0] * X[:,1], p5 = X[:,2]
Y_train = X_train.clone()
Y_train[:, 4] = X_train[:, 0] * X_train[:, 1]
Y_train[:, 5] = X_train[:, 2]

print('Training differentiable logic program...')
for epoch in range(50):
    pred = program(X_train, steps=2)
    loss = F.mse_loss(pred, Y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}')

# Test
with torch.no_grad():
    test_in = torch.tensor([[0.9, 0.8, 0.1, 0.5, 0.0, 0.0]])
    out = program(test_in, steps=2)
    print(f'\nInput:  {test_in[0].tolist()}')
    print(f'Output: {[round(v, 3) for v in out[0].tolist()]}')
    print(f'Expected p4 ~= {0.9*0.8:.3f}, p5 ~= 0.1')

print(f'\nKey: Differentiable logic makes symbolic rules trainable via gradient descent.')
print(f'Fuzzy t-norms provide smooth approximations of crisp logical operators.')

## 4. LLM + 符号推理

**核心动机**：LLM 擅长语言理解与生成，但在严格推理（数学、逻辑、代码执行）上易出错。结合外部符号工具可大幅提升可靠性。

**主要范式**：
- **Program-Aided Language Models (PAL)**：LLM 生成代码，由解释器执行得到答案
- **Tool-augmented reasoning**：LLM 调用计算器、检索器、求解器等工具
- **Chain-of-Thought + Verification**：LLM 生成推理链，符号验证器检查正确性
- **Neuro-Symbolic Solver**：LLM 提供候选解，符号系统选择与验证

**优势**：
- 计算与逻辑交给确定性工具，避免 LLM 的数值/逻辑错误
- 推理过程可追溯、可验证
- LLM 负责将自然语言转化为符号表示

**代表工作**：PAL、PoT (Program of Thoughts)、Toolformer、ReAct、FunSearch

In [ ]:
torch.manual_seed(42)

print('=== LLM + Symbolic Reasoning ===')


class TinyLLMSimulator(nn.Module):
    def __init__(self, vocab_size=200, d_model=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.rnn = nn.GRU(d_model, d_model, batch_first=True)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, tokens):
        emb = self.embed(tokens)
        out, _ = self.rnn(emb)
        return self.head(out)

    @torch.no_grad()
    def generate(self, prompt, max_len=16):
        tokens = prompt.clone()
        for _ in range(max_len):
            logits = self.forward(tokens)
            next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            tokens = torch.cat([tokens, next_tok], dim=1)
        return tokens


class SymbolicVerifier:
    def __init__(self):
        self.rules = {
            'add': lambda a, b: a + b,
            'sub': lambda a, b: a - b,
            'mul': lambda a, b: a * b,
            'div': lambda a, b: a / b if b != 0 else float('inf'),
        }

    def verify_program(self, program_str, expected_answer):
        try:
            # Safe eval of simple arithmetic expression
            allowed = set('0123456789+-*/(). ')
            if not set(program_str).issubset(allowed):
                return False, 'invalid characters'
            result = eval(program_str, {'__builtins__': {}}, {})
            return abs(result - expected_answer) < 1e-6, result
        except Exception as e:
            return False, str(e)


class ProgramAidedSolver:
    def __init__(self):
        self.verifier = SymbolicVerifier()

    def solve(self, problem):
        # problem: dict with 'numbers', 'ops', 'question', 'answer'
        # Generate candidate programs by enumerating op combinations
        numbers = problem['numbers']
        ops = problem['ops']
        target = problem['answer']

        candidates = []
        for op in ops:
            expr = f'{numbers[0]} {op} {numbers[1]}'
            candidates.append(expr)

        for expr in candidates:
            ok, result = self.verifier.verify_program(expr, target)
            if ok:
                return expr, result, True
        return candidates[0] if candidates else '', None, False


class LLMSymbolicReasoner:
    def __init__(self):
        self.llm = TinyLLMSimulator()
        self.solver = ProgramAidedSolver()

    def solve_problem(self, problem):
        # Step 1: LLM 'understands' the problem (simulated)
        prompt = torch.randint(0, 200, (1, 4))
        _ = self.llm.generate(prompt, max_len=8)  # simulate generation
        # Step 2: symbolic solver executes and verifies
        expr, result, ok = self.solver.solve(problem)
        return expr, result, ok


# Demo: math problem solving
reasoner = LLMSymbolicReasoner()

problems = [
    {'numbers': [15, 27], 'ops': ['+', '-', '*'], 'answer': 42, 'q': '15 + 27 = ?'},
    {'numbers': [8, 9], 'ops': ['+', '-', '*'], 'answer': 72, 'q': '8 * 9 = ?'},
    {'numbers': [100, 37], 'ops': ['+', '-', '*'], 'answer': 63, 'q': '100 - 37 = ?'},
]

print('Solving math problems with LLM + symbolic verifier:')
for p in problems:
    q_text = p['q']
    expr, result, ok = reasoner.solve_problem(p)
    status = 'VERIFIED' if ok else 'FAILED'
    print(f'  Q: {q_text}  =>  {expr} = {result}  [{status}]')

print(f'\nKey: LLM handles language understanding; symbolic tools handle exact computation.')
print(f'This separation avoids LLM numerical errors and produces verifiable results.')

## 5. 知识图谱与神经推理

**知识图谱（KG）**：以 (头实体, 关系, 尾实体) 三元组形式存储结构化知识，是符号知识的典型载体。

**神经 KG 推理方向**：
- **链接预测**：预测缺失的三元组（TransE、RotatE、ComplEx）
- **多跳推理**：在 KG 上做多步推理回答复杂问题
- **GNN 推理**：用图神经网络沿 KG 边传播信息

**神经符号结合**：
- KG 提供符号化的背景知识
- 神经模型学习实体/关系的嵌入表示
- 推理路径既可解释又可学习

**应用**：问答系统、推荐系统、医疗诊断、金融风控

In [ ]:
torch.manual_seed(42)

print('=== Knowledge Graph Neural Reasoning ===')


class KnowledgeGraphReasoner(nn.Module):
    def __init__(self, n_entities=20, n_relations=5, dim=32):
        super().__init__()
        self.entity_emb = nn.Embedding(n_entities, dim)
        self.relation_emb = nn.Embedding(n_relations, dim)
        nn.init.uniform_(self.entity_emb.weight, -0.1, 0.1)
        nn.init.uniform_(self.relation_emb.weight, -0.1, 0.1)

    def score(self, head, relation, tail):
        # TransE-style score: ||h + r - t||
        h = self.entity_emb(head)
        r = self.relation_emb(relation)
        t = self.entity_emb(tail)
        return -(h + r - t).norm(p=2, dim=-1)

    def link_prediction(self, head, relation, candidates):
        # Rank candidate tails for a given (head, relation)
        h = self.entity_emb(head).unsqueeze(1)
        r = self.relation_emb(relation).unsqueeze(1)
        t = self.entity_emb(candidates).unsqueeze(0)
        scores = -(h + r - t).norm(p=2, dim=-1)
        return scores

    def forward(self, head, relation, tail):
        return self.score(head, relation, tail)


class GraphNeuralReasoner(nn.Module):
    def __init__(self, n_entities=20, n_relations=5, dim=32):
        super().__init__()
        self.entity_emb = nn.Embedding(n_entities, dim)
        self.relation_emb = nn.Embedding(n_relations, dim)
        # Message passing layers
        self.msg_pass = nn.Linear(dim * 2, dim)
        self.update = nn.GRUCell(dim, dim)
        self.n_entities = n_entities

    def multi_hop_reason(self, start_entity, adjacency, n_hops=2):
        # adjacency: (n_entities, n_entities) binary matrix
        h = self.entity_emb.weight.clone()
        path_scores = torch.zeros(self.n_entities)
        path_scores[start_entity] = 1.0

        for hop in range(n_hops):
            # Propagate scores along edges
            new_scores = adjacency.t() @ path_scores
            # Combine with current via neural update
            msg = self.msg_pass(torch.cat([h, new_scores.unsqueeze(-1).expand_as(h)], dim=-1))
            h = self.update(msg, h)
            path_scores = F.softmax(new_scores, dim=-1)
        return h, path_scores


# Build a small KG
n_entities = 20
n_relations = 5
kg_reasoner = KnowledgeGraphReasoner(n_entities, n_relations, dim=32)
optimizer = torch.optim.AdamW(kg_reasoner.parameters(), lr=1e-2)

# Synthetic triples (head, relation, tail)
triples = torch.tensor([
    [0, 0, 1], [1, 0, 2], [2, 1, 3], [3, 2, 4],
    [4, 3, 5], [5, 0, 6], [6, 1, 7], [7, 2, 8],
    [0, 4, 9], [9, 0, 10], [10, 1, 11], [2, 3, 12],
    [12, 2, 13], [13, 0, 14], [14, 4, 15], [1, 2, 16],
    [16, 1, 17], [17, 3, 18], [18, 0, 19], [3, 4, 0],
])

# Negative sampling training
print('Training KG link prediction (TransE)...')
for epoch in range(50):
    pos = kg_reasoner.score(triples[:, 0], triples[:, 1], triples[:, 2])
    neg_tail = torch.randint(0, n_entities, triples.shape)
    neg = kg_reasoner.score(triples[:, 0], triples[:, 1], neg_tail[:, 2])
    # Margin loss
    loss = F.relu(1.0 + pos - neg).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}')

# Link prediction demo
with torch.no_grad():
    candidates = torch.arange(n_entities)
    head = torch.tensor([0])
    rel = torch.tensor([0])
    scores = kg_reasoner.link_prediction(head, rel, candidates)[0]
    top5 = scores.topk(5)
    print(f'\nLink prediction for (entity 0, relation 0):')
    print(f'  Top-5 candidates: {top5.indices.tolist()}')
    print(f'  Scores: {[round(s, 3) for s in top5.values.tolist()]}')

# Multi-hop reasoning with GNN
gnn_reasoner = GraphNeuralReasoner(n_entities, n_relations, dim=32)
adjacency = torch.zeros(n_entities, n_entities)
for h, _, t in triples.tolist():
    adjacency[h, t] = 1.0

with torch.no_grad():
    final_emb, path_scores = gnn_reasoner.multi_hop_reason(
        start_entity=0, adjacency=adjacency, n_hops=2
    )
    top_reachable = path_scores.topk(5)
    print(f'\nMulti-hop reasoning from entity 0 (2 hops):')
    print(f'  Reachable entities: {top_reachable.indices.tolist()}')
    print(f'  Path scores: {[round(s, 3) for s in top_reachable.values.tolist()]}')

print(f'\nKey: KG reasoning combines symbolic graph structure with learned embeddings.')
print(f'Link prediction fills missing knowledge; GNN multi-hop reasoning answers complex queries.')

## 📝 课后思考题

1. 神经符号 AI 相比纯神经网络或纯符号系统，分别解决了哪些核心问题？请举例说明。
2. 在可微逻辑中，乘积 t-norm（$a \cdot b$）与最小 t-norm（$\min(a,b)$）在梯度传播上有何差异？哪个更适合训练？
3. 如果让你设计一个 LLM + 符号推理的数学求解系统，你会如何分配 LLM 与符号工具的职责？如何处理 LLM 生成错误程序的情况？
4. 知识图谱的多跳推理中，如何平衡符号路径的可解释性与神经嵌入的表达能力？路径数过多时如何剪枝？